# Product Recommendation System Using Machine Learning
## B.Tech Machine Learning Mini Project Notebook

### Objectives:
1. **Exploratory Data Analysis (EDA)** on products and user ratings.
2. **Content-Based Recommendation Engine** using TF-IDF & Cosine Similarity.
3. **Collaborative Filtering Engine** using User-Item Matrix and Cosine User Similarity.
4. **Hybrid Recommender** using dynamic weighted score fusion.
5. **Offline Evaluation** with Precision@K, Recall@K, F1-Score, and Rating RMSE.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Set visual themes
sns.set_theme(style="whitegrid", palette="muted")
print("Libraries successfully imported!")

## 1. Data Ingestion & Preprocessing

In [ ]:
products_df = pd.read_csv('../data/products.csv')
ratings_df = pd.read_csv('../data/ratings.csv')

print(f"Total Products: {len(products_df)}")
print(f"Total Ratings: {len(ratings_df)}")
display(products_df.head(3))
display(ratings_df.head(3))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=ratings_df, x="rating", palette="Blues_d")
plt.title("Rating Distribution")

plt.subplot(1, 2, 2)
sns.countplot(data=products_df, y="category", palette="viridis")
plt.title("Product Categories")
plt.tight_layout()
plt.show()

## 3. Content-Based Recommendation Engine

In [ ]:
# Construct Content Soup (Product Name, Category, Brand, Description)
products_df['content_soup'] = (
    products_df['product_name'] + ' ' +
    products_df['category'] + ' ' +
    products_df['brand'] + ' ' +
    products_df['description']
)

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(products_df['content_soup'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")

In [ ]:
def get_content_recommendations(product_id, top_n=5):
    pid_idx_map = {pid: i for i, pid in enumerate(products_df['product_id'])}
    if product_id not in pid_idx_map:
        return products_df.sort_values(by='rating', ascending=False).head(top_n)
    
    idx = pid_idx_map[product_id]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    
    recommended_indices = [s[0] for s in sim_scores]
    results = products_df.iloc[recommended_indices].copy()
    results['similarity_score'] = [round(s[1], 4) for s in sim_scores]
    return results[['product_id', 'product_name', 'category', 'brand', 'price', 'similarity_score']]

# Test recommending similar products to Sony WH-1000XM5 (P101)
get_content_recommendations('P101', top_n=5)

## 4. Collaborative Filtering Engine

In [ ]:
user_item_matrix = ratings_df.pivot_table(index='user_id', columns='product_id', values='rating').fillna(0)
user_sim_matrix = pd.DataFrame(
    cosine_similarity(user_item_matrix),
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User-Item Rating Matrix:")
display(user_item_matrix.head(4))
print("User Similarity Matrix:")
display(user_sim_matrix.iloc[:4, :4])

## 5. Hybrid Recommendation Function

In [ ]:
def get_hybrid_recommendation(user_id, alpha=0.5, top_n=5):
    if user_id not in user_item_matrix.index:
        return products_df.sort_values(by='rating', ascending=False).head(top_n)
    
    user_ratings = user_item_matrix.loc[user_id]
    unrated_pids = user_ratings[user_ratings == 0].index
    
    # Collab scores
    similar_users = user_sim_matrix[user_id].drop(user_id)
    top_users = similar_users[similar_users > 0]
    
    scores = []
    for pid in unrated_pids:
        collab_score = user_item_matrix.loc[top_users.index, pid].mean() / 5.0 if not top_users.empty else 0.5
        content_score = 0.5  # placeholder or aggregated score
        hybrid_score = (alpha * collab_score) + ((1 - alpha) * content_score)
        scores.append((pid, hybrid_score))
        
    scores.sort(key=lambda x: x[1], reverse=True)
    top_pids = [s[0] for s in scores[:top_n]]
    return products_df[products_df['product_id'].isin(top_pids)]

get_hybrid_recommendation('U1001', alpha=0.6, top_n=5)